# 1. Escolha do Município e Obtenção dos Dados:

In [1]:
import folium
import pandas as pd
import numpy as np
import geopandas as gpd

from folium.plugins import MiniMap
from branca.colormap import LinearColormap
from folium.features import GeoJsonTooltip

# Leitura dos DataFrames de Ouro Branco (Votação das Sessões Eleitorais e Local de Votação dos Eleitorados):
df_voting_session = pd.read_csv('votacao_secao_2026_MG_Ouro_Branco.csv', sep=';', encoding='latin_1')
df_electorate_voting_location = pd.read_csv('eleitorado_local_votacao_2026_MG_Ouro_Branco.csv', sep=';', encoding='latin_1')

display(df_voting_session.head())

,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,...,NR_SECAO,CD_CARGO,DS_CARGO,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS,NR_LOCAL_VOTACAO,SQ_CANDIDATO,NM_LOCAL_VOTACAO,DS_LOCAL_VOTACAO_ENDERECO
0,26/10/2026,12:07:25,2026,2,Eleição Ordinária,2,6258,Eleição Geral Federal 2026,25/10/2026,F,...,184,1,Presidente,96,VOTO NULO,7,1104,-1,ESCOLA ESTADUAL LEVINDO DA COSTA CARVALHO,"RUA AMARANTINA, S/N"
1,26/10/2026,12:07:25,2026,2,Eleição Ordinária,2,6258,Eleição Geral Federal 2026,25/10/2026,F,...,171,1,Presidente,96,VOTO NULO,4,1090,-1,ESCOLA ESTADUAL IRACEMA DE ALMEIDA,"RUA SANTA OLIMPIA, S/N"
2,26/10/2026,12:07:25,2026,2,Eleição Ordinária,2,6258,Eleição Geral Federal 2026,25/10/2026,F,...,141,1,Presidente,96,VOTO NULO,5,1031,-1,CAMARA MUNICIPAL DE OURO BRANCO,"PRACA SAGRADO CORACOES, 200"
3,26/10/2026,12:07:25,2026,2,Eleição Ordinária,2,6258,Eleição Geral Federal 2026,25/10/2026,F,...,230,1,Presidente,96,VOTO NULO,11,1317,-1,ESCOLA MUNICIPAL GERALDO MARINO VIEIRA,"RUA EUCALIPTO , 312"
4,26/10/2026,12:07:25,2026,2,Eleição Ordinária,2,6258,Eleição Geral Federal 2026,25/10/2026,F,...,167,1,Presidente,96,VOTO NULO,8,1082,-1,ESCOLA ESTADUAL CONEGO LUIZ VIEIRA DA SILVA,"RUA JOAO CATARINA, S/N"


# 2. Limpeza, Filtragem e Unificação de Dados:

In [2]:
# Adquirindo a localização das sessões eleitorais:

# Padronizando as chaves geográficas, selecionando as colunas para obter a localização de cada seção nas zonas eleitorais:
df_location = df_electorate_voting_location[
    ['NR_TURNO', 'NR_ZONA', 'NR_SECAO', 'NR_LATITUDE', 'NR_LONGITUDE']
    ].drop_duplicates(subset=['NR_TURNO', 'NR_ZONA', 'NR_SECAO'])

# Convertendo as coordenadas para valores numéricos:
df_location['NR_LATITUDE'] = df_location['NR_LATITUDE'].astype(str).str.replace(',', '.')
df_location['NR_LONGITUDE'] = df_location['NR_LONGITUDE'].astype(str).str.replace(',', '.')

df_location['NR_LATITUDE'] = pd.to_numeric(df_location['NR_LATITUDE'], errors='coerce')
df_location['NR_LONGITUDE'] = pd.to_numeric(df_location['NR_LONGITUDE'], errors='coerce')

# Removendo coordenadas nulas ou com o valor -1:
df_location = df_location.dropna(subset=['NR_LATITUDE', 'NR_LONGITUDE'])
df_location = df_location[(df_location['NR_LATITUDE'] != -1) & (df_location['NR_LONGITUDE'] != -1)]

center = [-20.5233, -43.6914]

# DataFrame com todos os dados que serão utilizados:
df = df_voting_session[['NR_TURNO', 'CD_ELEICAO', 'NR_ZONA', 'NR_SECAO', 'CD_CARGO', 
                        'DS_CARGO', 'NR_VOTAVEL', 'NM_VOTAVEL', 'QT_VOTOS', 'NM_LOCAL_VOTACAO', 'DS_LOCAL_VOTACAO_ENDERECO']].copy()

# Fazendo o merge das colunas 'NR_LATITUDE' e 'NR_LONGITUDE' no DataFrame:
df = df.merge(df_location, on=['NR_TURNO', 'NR_ZONA', 'NR_SECAO'], how='left')

# Agrupando a contagem total de votos por candidato em cada seção eleitoral nos dois turnos:
total_vote_count_grouped = df.groupby(
    ['NR_TURNO', 'NR_ZONA', 'NR_SECAO', 'NR_VOTAVEL', 'NM_VOTAVEL', 'NM_LOCAL_VOTACAO', 
     'DS_LOCAL_VOTACAO_ENDERECO', 'NR_LATITUDE', 'NR_LONGITUDE'],
     as_index=False
    )['QT_VOTOS'].sum()

# 3. Mapeamento Graduado do 1º Turno:

In [3]:
# Filtrando os dados do primeiro turno:
df_1t = total_vote_count_grouped[
    total_vote_count_grouped['NR_TURNO'].astype(str) == '1'
].copy()

# Removendo votos brancos e nulos:
df_1t = df_1t[
    ~df_1t['NM_VOTAVEL'].str.contains('BRANCO|NULO', case=False)
].copy()

total_vote_candidates = (
    df_1t.groupby('NM_VOTAVEL')['QT_VOTOS']
    .sum()
    .sort_values(ascending=False)
)

candidates = total_vote_candidates.head(5).index.tolist()

display(total_vote_candidates.head(5))

# Organizando os votos por seção eleitoral e por candidato:
votes_1t = df_1t.pivot_table(
        index=['NR_ZONA', 'NR_SECAO'],
        columns='NM_VOTAVEL',
        values='QT_VOTOS',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

votes_1t.columns.name = None

# Adicionando as informações de localização de cada seção eleitoral:
df_info_location = df_1t[
    ['NR_ZONA', 'NR_SECAO', 'NM_LOCAL_VOTACAO', 
     'DS_LOCAL_VOTACAO_ENDERECO', 'NR_LATITUDE', 'NR_LONGITUDE']
     ].drop_duplicates(subset=['NR_ZONA', 'NR_SECAO'])

votes_1t = votes_1t.merge(df_info_location, on=['NR_ZONA', 'NR_SECAO'], how='left', validate='one_to_one')

# Calculando o total de votos válidos por seção eleitoral:
all_candidates = [c for c in df_1t['NM_VOTAVEL'].unique() if c in votes_1t.columns]

candidates_columns = all_candidates

votes_1t['TOTAL_VALIDOS'] = votes_1t[candidates_columns].sum(axis=1)

# Calculando o percentual de votos de cada candidato por seção eleitoral:
for candidate in candidates:
    votes_1t[f'PERCENTUAL_{candidate}'] = np.where(votes_1t['TOTAL_VALIDOS'] > 0, votes_1t[candidate] / votes_1t['TOTAL_VALIDOS'] * 100, 0)

display(votes_1t.head())

votes_1t.to_csv(
    'votos_1t.csv',
    index=False,
    encoding='utf-8'
)

palettes = [
    '#272D97',
    '#D60828',
    '#4C8790',
    '#D8942F',
    '#A0A00B'
]

# Associando uma cor aos candidatos:
candidates_palettes = {
    candidate: palettes[i % len(palettes)]
    for i, candidate in enumerate(candidates)
}

def create_map_by_candidate(df, candidate, palette):
    """Função utilizada para criar um mapa para cada candidato, utilizando a latitude e longitude das seções eleitorais."""

    df_map = df.dropna(subset=['NR_LATITUDE', 'NR_LONGITUDE']).copy()
    df_map = df_map[(df_map['NR_LATITUDE'] != -1) & (df_map['NR_LONGITUDE'] != -1)]

    map_1t = folium.Map(location=center, zoom_start=12, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors')

    max_percentage = df_map[f'PERCENTUAL_{candidate}'].max()

    scale = LinearColormap(colors=['#ffffff', palette], vmin=0, vmax=max_percentage if max_percentage > 0 else 1)
    scale.caption = f'Percentual de votos - {candidate} (%)'

    gdf = gpd.GeoDataFrame(df_map, geometry=gpd.points_from_xy(df_map['NR_LONGITUDE'], df_map['NR_LATITUDE']), crs='EPSG:4326')
    tooltip_style = """
        background-color: white;
        color: #333333;
        font-family: Arial;
        font-size: 13px;
        border-radius: 6px;
        padding: 8px;
    """

    geo_json_tooltip = folium.GeoJsonTooltip(fields=['NR_ZONA', 'NR_SECAO', f'PERCENTUAL_{candidate}'], 
                                             aliases=['Zona:', 'Seção:', f'Votos de {candidate} (%):'], 
                                             style=tooltip_style, 
                                             sticky=True, 
                                             localize=True
                                             )

    folium.GeoJson(
        gdf,
        name='Seções eleitorais',
        marker=folium.CircleMarker(
            radius=8,
            weight=1,
            fill=True,
            fill_opacity=0.85
        ),
        style_function=lambda feature, candidate=candidate, palette=palette: {
            'color': palette,
            'fillColor': scale(
                feature['properties'][f'PERCENTUAL_{candidate}']
            )
        },
        tooltip=geo_json_tooltip
    ).add_to(map_1t)
    scale.add_to(map_1t)

    MiniMap(toggle_display=True).add_to(map_1t)

    folium.LayerControl().add_to(map_1t)

    return map_1t

maps_1t = {}

for candidate in candidates:
    palette = candidates_palettes[candidate]

    map_1t = create_map_by_candidate(votes_1t, candidate, palette)
    maps_1t[candidate] = map_1t

    filename = candidate.lower().replace(" ", "_")
    output_path = f"mapa_1t_{filename}.html"

    map_1t.save(output_path)

NM_VOTAVEL
FLAVIO NANTES BOLSONARO              10891
LUIZ INÁCIO LULA DA SILVA            10187
AUGUSTO JORGE CURY                    1043
RENAN ANTONIO FERREIRA DOS SANTOS      619
RONALDO RAMOS CAIADO                   461
Name: QT_VOTOS, dtype: int64

,NR_ZONA,NR_SECAO,AUGUSTO JORGE CURY,CLARIANA ZACARKIM BARAO,FLAVIO NANTES BOLSONARO,HERTZ DA CONCEICAO DIAS,LEONARDO ALVES DE ARAUJO,LUIZ INÁCIO LULA DA SILVA,RENAN ANTONIO FERREIRA DOS SANTOS,ROMEU ZEMA NETO,...,NM_LOCAL_VOTACAO,DS_LOCAL_VOTACAO_ENDERECO,NR_LATITUDE,NR_LONGITUDE,TOTAL_VALIDOS,PERCENTUAL_FLAVIO NANTES BOLSONARO,PERCENTUAL_LUIZ INÁCIO LULA DA SILVA,PERCENTUAL_AUGUSTO JORGE CURY,PERCENTUAL_RENAN ANTONIO FERREIRA DOS SANTOS,PERCENTUAL_RONALDO RAMOS CAIADO
0,88,138,7,0,102,0,0,103,7,10,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,233,43.776824,44.206009,3.004292,3.004292,1.716738
1,88,139,10,0,105,0,0,92,9,4,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,229,45.851528,40.174672,4.366812,3.930131,3.930131
2,88,140,6,0,107,0,0,95,4,2,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,222,48.198198,42.792793,2.702703,1.801802,2.702703
3,88,141,13,0,135,0,0,110,6,2,...,CAMARA MUNICIPAL DE OURO BRANCO,"PRACA SAGRADO CORACOES, 200",-20.512249,-43.722183,271,49.815498,40.590406,4.797048,2.214022,1.845018
4,88,142,13,0,123,0,0,101,5,4,...,CAMARA MUNICIPAL DE OURO BRANCO,"PRACA SAGRADO CORACOES, 200",-20.512249,-43.722183,253,48.616601,39.920949,5.138340,1.976285,2.766798
